# Historical Digital Edition — Complete Workflow

End-to-end pipeline for extending the Digital Edition of `bsb11005578`.

**What this notebook does:**
1. Downloads remaining unprocessed page images (seq 103+) via IIIF
2. Processes them through Gemini (Fraktur OCR + Named Entity Recognition)
3. Geocodes Location entities via Nominatim (for Leaflet maps)
4. Renders new pages as V1-compatible HTML (matching the existing edition exactly)
5. Merges new pages into `Digital_Edition_V1.html` to produce the complete book

**Prerequisites:**
- `GEMINI_API_KEY` set in Colab Secrets (key icon, left sidebar)
- `Digital_Edition_V1.html` on your Google Drive
- (Optional) Previous processing results JSON on Drive

In [ ]:
# === 0. Clone Repository & Install Dependencies ===
import os

REPO_DIR = "/content/historical-digital-edition"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Maelkolb/historical-digital-edition.git {REPO_DIR}
    print("Repo cloned.")
else:
    print("Repo already present.")

%cd {REPO_DIR}
!pip install -q google-genai pillow tqdm requests python-dotenv google-api-python-client
print("Dependencies installed.")

## 1. Environment Setup

Mounts Google Drive and reads the Gemini API key from Colab Secrets.

In [ ]:
import os, sys, logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

sys.path.insert(0, "/content/historical-digital-edition")

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    api_key = userdata.get("GEMINI_API_KEY")
    IN_COLAB = True
    print("Colab environment detected. Drive mounted.")
except ImportError:
    IN_COLAB = False
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.environ.get("GEMINI_API_KEY")
    print("Local environment detected.")

if not api_key:
    raise ValueError("Set GEMINI_API_KEY in Colab Secrets (key icon) or .env file.")

from google import genai
client = genai.Client(api_key=api_key)
print("Gemini client ready.")

## 2. Configure Paths

Edit these paths to match your Google Drive folder structure.

- **`NEW_IMAGE_FOLDER`** — where newly downloaded images are stored (seq 103+)
- **`NEW_OUTPUT_FOLDER`** — where new JSON results are saved (separate from old)
- **`PREVIOUS_JSON`** — path to the combined JSON from your first processing run (pages 3–90)
- **`V1_HTML_PATH`** — path to the existing `Digital_Edition_V1.html`
- **`DRIVE_IMAGES`** — Drive folder where all facsimile images are stored

In [ ]:
from pathlib import Path
from src import config

BOOK_ID        = config.BOOK_ID          # "bsb11005578"
MODEL_ID       = config.MODEL_ID
THINKING_LEVEL = config.THINKING_LEVEL
ENTITY_TYPES   = config.ENTITY_TYPES

if IN_COLAB:
    DRIVE_BASE = Path("/content/drive/MyDrive/digital_edition")
else:
    DRIVE_BASE = Path("..").resolve()

# New images go into a dedicated subfolder to avoid mixing with already-processed ones
NEW_IMAGE_FOLDER  = DRIVE_BASE / "images_new"
NEW_OUTPUT_FOLDER = DRIVE_BASE / "output_new"

# Previous results (pages 3-90) from your first processing run
PREVIOUS_JSON = DRIVE_BASE / "output" / "digital_edition_complete.json"

# The existing V1 HTML edition to extend
V1_HTML_PATH = DRIVE_BASE / "Digital_Edition_V1.html"

# Google Drive folder where ALL facsimile images are stored
DRIVE_IMAGES = DRIVE_BASE / "images"

# Final output
FINAL_OUTPUT = DRIVE_BASE / "digital_edition_complete.html"

# IIIF sequence range for remaining pages
START_SEQ = 103  # first unprocessed page
END_SEQ   = None  # None = download to end of book

NEW_IMAGE_FOLDER.mkdir(parents=True, exist_ok=True)
NEW_OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

print(f"New images     : {NEW_IMAGE_FOLDER}")
print(f"New output     : {NEW_OUTPUT_FOLDER}")
print(f"Previous JSON  : {PREVIOUS_JSON} {'(exists)' if PREVIOUS_JSON.exists() else '(not found)'}")
print(f"V1 HTML        : {V1_HTML_PATH} {'(exists)' if V1_HTML_PATH.exists() else '(not found)'}")
print(f"Drive images   : {DRIVE_IMAGES}")
print(f"Model          : {MODEL_ID}")
print(f"IIIF range     : seq {START_SEQ} to {'end' if END_SEQ is None else END_SEQ}")

## 3. Download Remaining Images (IIIF)

Downloads the unprocessed pages from the Bayerische Staatsbibliothek IIIF API.
Files are saved with names like `bsb11005578_seq_103_page_91_(0103).jpg`.

In [ ]:
from src.downloader import IIIFDownloader

downloader = IIIFDownloader(
    book_id=BOOK_ID,
    output_dir=NEW_IMAGE_FOLDER,
    delay=0.5,
)

downloaded = downloader.download(start_seq=START_SEQ, end_seq=END_SEQ)
print(f"\nDownloaded {len(downloaded)} images to {NEW_IMAGE_FOLDER}")

## 4. Copy Images to Google Drive

The V1 edition loads facsimile images from Google Drive. Copy the newly
downloaded images to the same Drive folder that holds the existing images
(pages 3–90).

If you downloaded directly to a Drive-mounted path in Section 3, also copy
them to the shared images folder so the Drive API can find them in Section 8.

In [ ]:
import shutil

DRIVE_IMAGES.mkdir(parents=True, exist_ok=True)
copied = 0

for f in downloaded:
    dest = DRIVE_IMAGES / f.name
    if not dest.exists():
        shutil.copy2(f, dest)
        copied += 1

print(f"Copied {copied} new images to {DRIVE_IMAGES}")
print(f"Total images in folder: {len(list(DRIVE_IMAGES.glob('*.jpg')))}")

## 5. Process New Pages (OCR + NER)

Runs the two-stage Gemini pipeline on each downloaded image:
- **Stage 1 — OCR:** Transcribes Fraktur text into structured JSON blocks
- **Stage 2 — NER:** Annotates named entities (Location, Person, Animal, etc.)

Results are saved individually as `page_NNNN.json` so that a crash can be
resumed by simply re-running this cell — already-processed pages are skipped.

In [ ]:
import json
from tqdm.notebook import tqdm
from src.pipeline import get_image_files, extract_page_number, process_page

json_folder = NEW_OUTPUT_FOLDER / "json"
json_folder.mkdir(parents=True, exist_ok=True)

image_files = get_image_files(NEW_IMAGE_FOLDER)
print(f"Found {len(image_files)} images in {NEW_IMAGE_FOLDER}")

new_results = []
skipped = 0
errors = []

for image_path in tqdm(image_files, desc="Processing pages"):
    page_num = extract_page_number(image_path.name)
    json_path = json_folder / f"page_{page_num:04d}.json"

    # Resume support: skip already-processed pages
    if json_path.exists():
        skipped += 1
        continue

    try:
        result = process_page(
            client=client,
            image_path=image_path,
            page_number=page_num,
            entity_types=ENTITY_TYPES,
            model_id=MODEL_ID,
            thinking_level=THINKING_LEVEL,
        )
        new_results.append(result)

        with open(json_path, "w", encoding="utf-8") as fh:
            json.dump(result.to_dict(), fh, ensure_ascii=False, indent=2)

    except Exception as exc:
        print(f"ERROR on {image_path.name}: {exc}")
        errors.append((image_path.name, str(exc)))

print(f"\nProcessed: {len(new_results)} | Skipped (already done): {skipped} | Errors: {len(errors)}")
if errors:
    print("Failed pages:", [e[0] for e in errors])

## 6. Load All Results & Merge with Previous Run

Loads all new page results from JSON (including any processed in earlier runs
of this notebook that were skipped above), then merges them with the pages
3–90 from the first processing run.

After this cell:
- `new_results_all` — all newly processed pages (91+)
- `all_results` — complete book (pages 3–90 + new pages)

In [ ]:
from src.pipeline import load_results_from_json, merge_results

# Load ALL newly processed pages from JSON (covers resume-skipped pages too)
new_results_all = load_results_from_json(json_folder)
print(f"New pages loaded: {len(new_results_all)}")
if new_results_all:
    print(f"  Page range: {new_results_all[0].page_number} - {new_results_all[-1].page_number}")
    print(f"  Entities: {sum(len(r.entities) for r in new_results_all)}")

# Load previous results (pages 3-90 from the first run)
if PREVIOUS_JSON.exists():
    previous_results = load_results_from_json(PREVIOUS_JSON)
    print(f"\nPrevious pages loaded: {len(previous_results)}")
else:
    print(f"\nWARNING: Previous JSON not found at {PREVIOUS_JSON}")
    print("Continuing with new pages only.")
    previous_results = []

# Merge: earlier lists have lower priority for duplicate page numbers
all_results = merge_results(previous_results, new_results_all)
print(f"\nCombined: {len(all_results)} total pages")
if all_results:
    print(f"  Full range: {all_results[0].page_number} - {all_results[-1].page_number}")
    print(f"  Total entities: {sum(len(r.entities) for r in all_results)}")

## 7. Geocode Locations via Nominatim

Resolves Location entities to geographic coordinates using OpenStreetMap
Nominatim (1 request/second, no API key needed). A cache dict avoids
re-querying the same place name twice.

Only **new** pages are geocoded — the V1 HTML already contains map data
for pages 3–90.

After this cell:
- `geo_cache` — dict mapping location name to coordinates
- `new_map_data` — dict mapping page number to Leaflet map data

In [ ]:
from src.geocoding import geocode_entities, build_page_map_data

# Collect all entities from newly processed pages
all_new_entities = [e for r in new_results_all for e in r.entities]
location_names = list(dict.fromkeys(
    e.text for e in all_new_entities if e.entity_type == "Location"
))
print(f"Unique location names to geocode: {len(location_names)}")
print(f"Estimated time: ~{len(location_names)} seconds (1 req/sec)")

# Geocode with cache
geo_cache = geocode_entities(all_new_entities, cache={}, delay=1.0)

geocoded = sum(1 for v in geo_cache.values() if v is not None)
failed = sum(1 for v in geo_cache.values() if v is None)
print(f"\nGeocoded: {geocoded}/{len(geo_cache)} locations ({failed} not found)")

# Build per-page map data for Leaflet
new_map_data = {}
for r in new_results_all:
    entry = build_page_map_data(r.page_number, r.entities, geo_cache)
    if entry is not None:
        new_map_data[r.page_number] = entry

print(f"Pages with map data: {len(new_map_data)}/{len(new_results_all)}")

## 8. Build Google Drive Image Manifest

The V1 edition loads facsimile images from Google Drive via
`https://lh3.googleusercontent.com/d/{fileId}`. This cell queries the
Drive API to find file IDs for all BSB images.

**Prerequisites:** Images must be uploaded to Google Drive (done in Section 4).

In [ ]:
import re

new_image_manifest = {}

if IN_COLAB:
    from googleapiclient.discovery import build as build_service
    from google.colab import auth
    auth.authenticate_user()
    drive_service = build_service("drive", "v3")

    query = f"name contains '{BOOK_ID}_seq_' and mimeType='image/jpeg'"
    page_token = None
    total_found = 0

    # Paginate through all results (handles >200 images)
    while True:
        kwargs = {
            "q": query,
            "fields": "nextPageToken, files(id, name)",
            "pageSize": 200,
        }
        if page_token:
            kwargs["pageToken"] = page_token

        resp = drive_service.files().list(**kwargs).execute()

        for f in resp.get("files", []):
            m = re.search(r"page[_-]?(\d+)", f["name"])
            if m:
                new_image_manifest[m.group(1)] = f["id"]
                total_found += 1

        page_token = resp.get("nextPageToken")
        if not page_token:
            break

    print(f"Found {len(new_image_manifest)} image file IDs on Drive")
else:
    # Local: provide file IDs manually
    # new_image_manifest = {"91": "DRIVE_FILE_ID", "92": "DRIVE_FILE_ID", ...}
    print("Not on Colab - populate new_image_manifest manually.")

# Preview first 5 entries
for pn, fid in sorted(new_image_manifest.items(), key=lambda x: int(x[0]))[:5]:
    print(f"  Page {pn}: {fid}")

## 9. Render V1-Compatible Pages & Merge into Edition

For each newly processed page:
1. Renders a V1-compatible `<article>` (entity highlights, Leaflet map,
   GBIF links, bilingual DE/EN, Drive facsimile)
2. Injects all new articles into `Digital_Edition_V1.html`
3. Merges new entries into `pageMapData` (Leaflet) and `imageManifest` (Drive)
4. Updates sidebar stats (page count, annotation count)

The output is the **complete combined edition** with all pages of the book.

In [ ]:
from src.v1_renderer import render_v1_page
from src.v1_merger import merge_into_v1

# Pre-flight checks
assert V1_HTML_PATH.exists(), f"V1 HTML not found: {V1_HTML_PATH}"
assert new_results_all, "No new results to render - did Section 6 complete?"

# Render each new page as a V1-compatible <article>
new_articles = []
for r in new_results_all:
    pn_str = str(r.page_number)
    article = render_v1_page(
        result=r,
        map_data=new_map_data.get(r.page_number),
        drive_file_id=new_image_manifest.get(pn_str),
    )
    new_articles.append(article)

print(f"Rendered {len(new_articles)} V1-compatible page articles")

# Merge into existing V1 HTML
output_file = merge_into_v1(
    v1_html_path=V1_HTML_PATH,
    new_page_articles=new_articles,
    new_map_data=new_map_data,
    new_image_manifest=new_image_manifest,
    output_path=FINAL_OUTPUT,
)

print(f"\nCombined edition: {output_file}")
print(f"File size: {output_file.stat().st_size / 1e6:.1f} MB")

## 10. Export Entities to CSV

Exports all entities (old + new) for further analysis.

In [ ]:
import csv

csv_path = NEW_OUTPUT_FOLDER / "all_entities.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["page", "entity_type", "text", "context", "start_char", "end_char"])
    for r in all_results:
        for e in r.entities:
            writer.writerow([r.page_number, e.entity_type, e.text, e.context or "", e.start_char, e.end_char])

total_ents = sum(len(r.entities) for r in all_results)
print(f"Exported {total_ents} entities to {csv_path}")

# Top 20 most frequent entities
freq = {}
for r in all_results:
    for e in r.entities:
        freq[(e.entity_type, e.text)] = freq.get((e.entity_type, e.text), 0) + 1

print("\nTop 20 entities:")
for (etype, text), count in sorted(freq.items(), key=lambda x: -x[1])[:20]:
    print(f"  [{etype}] \"{text}\": {count}x")

## 11. Download Final Edition

Downloads the completed combined HTML file. It is also saved on your
Drive at the path shown above.

In [ ]:
import re as _re

# Summary
final_html = output_file.read_text(encoding="utf-8")
total_pages = len(_re.findall(r'class="page-article"', final_html))
total_annotations = len(_re.findall(r'class="entity"', final_html))

print("=== Final Edition Summary ===")
print(f"  Pages       : {total_pages}")
print(f"  Annotations : {total_annotations:,}")
print(f"  File size   : {output_file.stat().st_size / 1e6:.1f} MB")
print(f"  Path        : {output_file}")

if IN_COLAB:
    from google.colab import files
    files.download(str(output_file))
    print("\nDownload started in your browser.")
else:
    print(f"\nFile is at: {output_file}")